Feature Eng -> Feature Transformation -> Handling categorical data -> One Hot Encoding

In [20]:
import numpy as np
import pandas as pd

In [21]:
df = pd.read_csv('cars.csv')

In [22]:
df.head()

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


In [23]:
# nominal categorical: brand, fuel, owner
# selling_price : o/p 

In [6]:
df['brand'].nunique()   # number of unique categories in brands column

32

In [24]:
df['owner'].value_counts()

owner
First Owner             5289
Second Owner            2105
Third Owner              555
Fourth & Above Owner     174
Test Drive Car             5
Name: count, dtype: int64

## 1. OneHotEncoding using Pandas

get_dummies

In [25]:
pd.get_dummies(df,columns=['fuel','owner']) 
# Not taking brand coz it has 32 unique categories, so it will create 32 new columns 
# which will create difficulty in understanding the concept for now.

,brand,km_driven,selling_price,fuel_CNG,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_First Owner,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,False,True,False,False,True,False,False,False,False
1,Skoda,120000,370000,False,True,False,False,False,False,True,False,False
2,Honda,140000,158000,False,False,False,True,False,False,False,False,True
3,Hyundai,127000,225000,False,True,False,False,True,False,False,False,False
4,Maruti,120000,130000,False,False,False,True,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,False,False,False,True,True,False,False,False,False
8124,Hyundai,119000,135000,False,True,False,False,False,True,False,False,False
8125,Maruti,120000,382000,False,True,False,False,True,False,False,False,False
8126,Tata,25000,290000,False,True,False,False,True,False,False,False,False


In [26]:
# previous cols = 5 , after encoding cols = 5 + 3 + 4 = 12 cols

In [27]:
# but we have to remove 1 col from fuel and 1 col from owner to avoid dummy variable trap. ( Multicollinearity problem)

## 2. K-1 OneHotEncoding

In [28]:
pd.get_dummies(df,columns=['fuel','owner'],drop_first=True) 
# fuel_CNG and owner_First Owner are removed.

,brand,km_driven,selling_price,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,True,False,False,False,False,False,False
1,Skoda,120000,370000,True,False,False,False,True,False,False
2,Honda,140000,158000,False,False,True,False,False,False,True
3,Hyundai,127000,225000,True,False,False,False,False,False,False
4,Maruti,120000,130000,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,False,False,True,False,False,False,False
8124,Hyundai,119000,135000,True,False,False,True,False,False,False
8125,Maruti,120000,382000,True,False,False,False,False,False,False
8126,Tata,25000,290000,True,False,False,False,False,False,False


Note : In ML we don't do One-Hot-Encoding using pandas, coz it doesn't remember the order of colunms so it can give different results everytime, So we use Sklearn.

## 3. OneHotEncoding using Sklearn

Train-Test Split

In [29]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.iloc[:,0:4],df.iloc[:,-1],test_size=0.2,random_state=2)

In [32]:
X_train.head() # only i/p cols

,brand,km_driven,fuel,owner
5571,Hyundai,35000,Diesel,First Owner
2038,Jeep,60000,Diesel,First Owner
2957,Hyundai,25000,Petrol,First Owner
7618,Mahindra,130000,Diesel,Second Owner
6684,Hyundai,155000,Diesel,First Owner


In [33]:
X_test.head() # only i/p cols

,brand,km_driven,fuel,owner
606,Hyundai,80000,Petrol,First Owner
7575,Mahindra,70000,Diesel,Second Owner
7705,Toyota,68089,Petrol,First Owner
4305,Hyundai,70000,Petrol,Second Owner
2685,Mahindra,97000,Diesel,Second Owner


OneHotEncoder

In [34]:
from sklearn.preprocessing import OneHotEncoder

In [35]:
# Teda Kaam hai coz haam fuel aur owner par OneHotEncoding kar rahe  hai and brand par nahi kar rahe hai.
# So Humey pheley fuel aur owner par encoding karna padega, uske baad brand & km_driven mey merge karna hoga.
# Eventually  hum iskey liye ColumnTransformer use kartey hai but abhi ke liye hum manually kartey hai.

In [41]:
ohe = OneHotEncoder(drop='first',sparse_output=False,dtype=np.int32) 
# drop = 'first' is used to avoid dummy variable trap
# sparse_output=False is used to get output in array format instead of sparse matrix format
# dtype=np.int32 is used to get output in integer format instead of float format.

In [42]:
X_train_new = ohe.fit_transform(X_train[['fuel','owner']]) # fit & transform on train data in one step

In [43]:
X_test_new = ohe.transform(X_test[['fuel','owner']])

In [44]:
X_train_new.shape

(6502, 7)

In [48]:
# To merge X_train_new with brand and km_driven columns, we can use np.hstack() horizontal stacking function.

In [49]:
np.hstack((X_train[['brand','km_driven']].values,X_train_new))

array([['Hyundai', 35000, 1, ..., 0, 0, 0],
       ['Jeep', 60000, 1, ..., 0, 0, 0],
       ['Hyundai', 25000, 0, ..., 0, 0, 0],
       ...,
       ['Tata', 15000, 0, ..., 0, 0, 0],
       ['Maruti', 32500, 1, ..., 1, 0, 0],
       ['Isuzu', 121000, 1, ..., 0, 0, 0]], shape=(6502, 9), dtype=object)

In [62]:
# We don't have to do np.hstack() after learning ColumnTransformer.

## 4. OneHotEncoding with Top Categories

In [50]:
counts = df['brand'].value_counts()

In [51]:
counts

brand
Maruti           2448
Hyundai          1415
Mahindra          772
Tata              734
Toyota            488
Honda             467
Ford              397
Chevrolet         230
Renault           228
Volkswagen        186
BMW               120
Skoda             105
Nissan             81
Jaguar             71
Volvo              67
Datsun             65
Mercedes-Benz      54
Fiat               47
Audi               40
Lexus              34
Jeep               31
Mitsubishi         14
Land                6
Force               6
Isuzu               5
Ambassador          4
Kia                 4
MG                  3
Daewoo              3
Ashok               1
Opel                1
Peugeot             1
Name: count, dtype: int64

In [54]:
threshold = 100           # humen 100 car ka threshold decide kiya hai, jiske niche aane wale brands ko hum 'other' category mey daal denge.

In [55]:
repl = counts[counts <= threshold].index

In [57]:
repl # name of brand which has count less than or equal to 100

Index(['Nissan', 'Jaguar', 'Volvo', 'Datsun', 'Mercedes-Benz', 'Fiat', 'Audi',
       'Lexus', 'Jeep', 'Mitsubishi', 'Land', 'Force', 'Isuzu', 'Ambassador',
       'Kia', 'MG', 'Daewoo', 'Ashok', 'Opel', 'Peugeot'],
      dtype='object', name='brand')

In [58]:
pd.get_dummies(df['brand'].replace(repl, 'uncommon')).sample(5)

,BMW,Chevrolet,Ford,Honda,Hyundai,Mahindra,Maruti,Renault,Skoda,Tata,Toyota,Volkswagen,uncommon
7992,False,False,False,False,False,False,True,False,False,False,False,False,False
1654,False,False,False,False,True,False,False,False,False,False,False,False,False
1739,False,False,False,False,True,False,False,False,False,False,False,False,False
5381,False,False,False,False,True,False,False,False,False,False,False,False,False
6705,False,False,False,True,False,False,False,False,False,False,False,False,False


In [60]:
# Note : True/False are not string they are boolean values

In [61]:
# So we succesfully converted categorical to numerical